## Incremento de la Resolución Semántica (K=13)

El  objetivo de este cuaderno es evaluar si el incremento del número de clústeres a $k=13$ permite resolver las ambigüedades detectadas en el modelo de 11 clústeres, logrando separar dimensiones informativas críticas (como el inicio del tratamiento o instrucciones de formas farmacéuticas específicas) que antes aparecían diluidas.

In [1]:
import pandas as pd
df_parrafos = pd.read_csv("prospectos_limpio_segmentado.csv")

In [2]:
df_parrafos = df_parrafos.drop_duplicates(subset=['parrafo_anonimizado'])

### Hipótesis de la "Fisión Informativa"
Tras observar que en el modelo de $k=11$ temas vitales como el embarazo o la conducción no lograban una independencia vectorial clara, se propone escalar a **13 clústeres**. 

Técnicamente, buscamos que el algoritmo tenga la capacidad de "romper" los macro-grupos inestables y rescatar sub-temas de alto valor para el paciente que el modelo de 11 clústeres consideraba como ruido o información secundaria.

In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

# Cargar modelo de embeddings (multilingüe para español)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 1. Limpiar y vectorizar
textos = df_parrafos['parrafo_anonimizado'].tolist()
embeddings = model.encode(textos, show_progress_bar=True)

# 2. Clustering
num_clusters = 13 
clustering_model = KMeans(n_clusters=num_clusters, random_state=42)
df_parrafos['cluster'] = clustering_model.fit_predict(embeddings)

c:\Users\04jul\Desktop\CDIA\TFG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 204.20it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 969/969 [24:10<00:00,  1.50s/it]


In [4]:
df_parrafos.to_csv("prospectos_clusters13.csv", index=False)

### Preparación de prompts

Seguimos el mismo procedimiento que para 11 clusters.

In [5]:
def generar_prompt_final(cluster_id, ejemplos):
  ejemplos_str = "\n- ".join(ejemplos)
      
      # PROMPT REFINADO: Foco absoluto en el usuario final
  prompt = f"""[INST] <<SYS>>
  Eres un PACIENTE que tiene una duda urgente y busca la respuesta en estos fragmentos. 
  TU REGLA MÁXIMA: Escribe ÚNICAMENTE la pregunta que harías a tu médico.
  PROHIBIDO usar palabras como: 'texto', 'fragmento', 'prospecto', 'diseño', 'JSON', 'información', 'sección', 'analizar'.
  PROHIBIDO mencionar nombres de medicamentos.
  <</SYS>>
  Sigue este razonamiento interno:
  1. Este fragmento "Puede disminuir la capacidad para conducir o manejar maquinaria por contener propilenglicol, ya que puede producir síntomas parecidos a los del alcohol." responde a la pregunta "¿El paracetamol afecta a la capacidad de conducción?".
  2. Este fragmento "Estas dosis se pueden repetir cada 4 horas." no responde a esa pregunta, responde a "¿Cada cuánto tiempo me lo tengo que tomar?".
  3. Teniendo esto en cuenta, ¿qué PREGUNTA UNIVERSAL responden todos los fragmentos del grupo?
  Basándote en estos fragmentos:
  {ejemplos_str}

  Genera la PREGUNTA DIRECTA que un paciente normal (no experto) haría. 
  Ejemplos de estilo correcto: 
  - "¿Qué hago si se me olvida una dosis?" 
  - "¿Puedo conducir después de tomar esto?"
  - "¿Dónde debo guardar la medicina?"

  Responde exclusivamente en este formato JSON:
  {{
    "pregunta_del_paciente": "Escribe aquí la pregunta"
  }}
  [/INST]"""
  return prompt

## Conexión con Ollama

In [6]:
import requests
import json
import pandas as pd

def consultar_ollama_stream(prompt, modelo_id):
    url = "https://wiig.dia.fi.upm.es/ollama/v1/chat/completions"
    payload = {
        "model": modelo_id,
        "messages": [{"role": "user", "content": prompt}],
        "stream": True
    }
    
    texto_acumulado = ""
    try:
        with requests.post(url, json=payload, stream=True, timeout=150) as r:
            r.raise_for_status()
            for line in r.iter_lines():
                if not line: continue
                decoded = line.decode("utf-8").replace("data: ", "").strip()
                if decoded == "[DONE]": break
                try:
                    chunk = json.loads(decoded)
                    content = chunk["choices"][0].get("delta", {}).get("content", "")
                    texto_acumulado += content
                except: continue
        return texto_acumulado
    except Exception as e:
        return f"ERROR: {str(e)}"

In [8]:
anotadores = ["llama3:latest", "llama3_hard:latest", "llama3_medium:latest"]
resultados = []

In [ ]:
import time

for c_id in sorted(df_parrafos['cluster'].unique()):
    fila = {}
    print(f" Procesando Cluster {c_id}...")
    
    # 1. Selección de variedad
    df_c = df_parrafos[df_parrafos['cluster'] == c_id]
    ejemplos_variados = df_c["parrafo_anonimizado"].to_list()
    
    # 2. Generación del prompt
    prompt_texto = generar_prompt_final(c_id, ejemplos_variados)
    
    fila = {"cluster": c_id, "ejemplos_usados": "|".join(ejemplos_variados)}
    
    # 3. Consulta a los LLMs
    for modelo in anotadores:
        print(f"  -> Consultando {modelo}...")
        respuesta = consultar_ollama_stream(prompt_texto, modelo)
        print(respuesta)
        fila[f"res_{modelo.replace(':', '_')}"] = respuesta
        time.sleep(1) # Pausa para no saturar el servidor

    resultados.append(fila)

df_final = pd.DataFrame(resultados)
df_final.to_csv("resultados_etiquetado_13topicos.csv", index=False)

 Procesando Cluster 0...
  -> Consultando llama3:latest...
{
"pregunta_del_paciente": "¿Cuál es el riesgo de coágulos sanguíneos con este medicamento y cómo puedo reducir ese riesgo?"
}
  -> Consultando llama3_hard:latest...
{
"pregunta_del_paciente": "¿Qué hago si tengo problemas para moverme durante el tratamiento y me duele el abdomen?" 
}
  -> Consultando llama3_medium:latest...
{
"pregunta_del_paciente": "¿Qué hago si siento dolor pélvico después de tomar [MEDICAMENTO]? ¿Es normal? ¿Puedo seguir tomando el medicamento?"
}
 Procesando Cluster 1...
  -> Consultando llama3:latest...
{
"pregunta_del_paciente": "¿Puedo dejar mi niño cerca del producto porque es muy sensible al grupo de medicamentos a los cuales pertenece hidroxicloroquina?"
  -> Consultando llama3_hard:latest...
{ "pregunta_del_paciente": "¿Puedo tomar este medicamento si tengo alergia a hidroxicloroquina?" }
  -> Consultando llama3_medium:latest...
{
"pregunta_del_paciente": "¿Puedo tomar otros medicamentos al mismo t

###  Evaluación de Resultados 
El análisis de las preguntas generadas por el comité de modelos de lenguaje confirma una evolución técnica significativa respecto a la iteración anterior. Se observan tres grandes victorias en la organización de la información:

1. **Identificación del Inicio del Tratamiento (Cluster 3):** Se ha logrado aislar el momento crítico de la toma inicial (ej: "¿Cuándo puedo empezar con la primera tira del blíster?").
2. **Instrucciones de Formas Farmacéuticas Específicas (Cluster 5):** El modelo ha segregado con éxito instrucciones mecánicas complejas, como el uso de supositorios o inhaladores (ej: "¿Cómo debo usar el supositorio correctamente?").
3. **Interacciones con el Estilo de Vida (Cluster 10):** Aparece por primera vez un bloque dedicado al consumo de alcohol y problemas visuales específicos, dotando al sistema de una capa de seguridad más "médica".

In [ ]:
import pandas as pd
import json
import re

def extraer_pregunta_segura(texto):
    """
    Limpia y extrae la pregunta del JSON generado por el LLM, 
    gestionando texto basura y fallos de formato.
    """
    if pd.isna(texto) or "ERROR" in str(texto):
        return "Error en respuesta"

    # Busca las llaves del JSON
    match = re.search(r'\{.*\}', str(texto), re.DOTALL)
    
    if match:
        json_str = match.group()
        try:
            # Reemplazar comillas dobles repetidas 
            json_str = json_str.replace('""', '"')
            datos = json.loads(json_str)
            
            # Buscamos la pregunta en las posibles llaves que el LLM haya inventado
            pregunta = datos.get('pregunta_final')
            
            if pregunta:
                return pregunta.strip()
        except Exception:
            pass

    # Si falla el JSON, buscamos patrones de texto 
    # Busca frases que empiecen por "¿" y terminen por "?"
    preguntas_encontradas = re.findall(r'¿[^?]+\?', str(texto))
    if preguntas_encontradas:
        # Devolvemos la última pregunta encontrada (suele ser la "final")
        return preguntas_encontradas[-1].strip()

    return "No se pudo identificar la pregunta"


df_resultados = pd.read_csv('resultados_etiquetado_13topicos.csv')

anotadores = [col for col in df_resultados.columns if col.startswith('res_')]

for col in anotadores:
    # Creamos una columna limpia para cada modelo
    nombre_limpio = col.replace('res_', 'pregunta_')
    df_resultados[nombre_limpio] = df_resultados[col].apply(extraer_pregunta_segura)


columnas_formulario = ['cluster'] + [col for col in df_resultados.columns if col.startswith('pregunta_')]
print(df_resultados[columnas_formulario].head())


df_resultados[columnas_formulario].to_csv('preguntas_para_formulario_13.csv', index=False)

   cluster                             pregunta_llama3_latest  \
0        0  ¿Cuál es el riesgo de coágulos sanguíneos con ...   
1        1  ¿Puedo dejar mi niño cerca del producto porque...   
2        2  ¿Debo hablar con mi médico antes de empezar a ...   
3        3  ¿Cuándo puedo empezar con la primera tira del ...   
4        4  ¿Debo tomar dosis dobles para compensar la dos...   

                         pregunta_llama3_hard_latest  \
0  ¿Qué hago si tengo problemas para moverme dura...   
1  ¿Puedo tomar este medicamento si tengo alergia...   
2                                   ¿qué debo hacer?   
3  ¿Es importante seguir el horario de toma o pue...   
4                     ¿Qué hago si olvido una dosis?   

                       pregunta_llama3_medium_latest  
0              ¿Puedo seguir tomando el medicamento?  
1  ¿Puedo tomar otros medicamentos al mismo tiemp...  
2                                         ¿qué hago?  
3  ¿Cuándo puedo empezar con la primera tira del ...

###  Diagnóstico de Limitaciones
A pesar de la ganancia en resolución, el modelo de 13 clústeres revela un fenómeno persistente derivado de la redacción original de la AEMPS:

* **Repetición Crónica:** El tema del "Olvido de dosis" se ha fragmentado en lugar de unificarse, apareciendo en los clústeres 4, 6, 7, 8, 10 y 12. Esto se debe a que el contenido original repite estas instrucciones en múltiples contextos (embarazo, parches, etc.), y el algoritmo las separa por contexto léxico en lugar de por intención funcional.
* **Persistencia de Ruido:** El **Cluster 2** sigue mostrando una calidad pobre, generando preguntas vacuas como "¿Qué debo hacer?", lo que indica la presencia de párrafos de relleno legal que aún no han sido aislados.
* **Cajones de Sastre:** El **Cluster 12** mezcla de forma confusa casos de sobredosis con olvidos de toma, impidiendo la generación de una pregunta maestra clara.

**Conclusión:** Aunque el modelo de 13 clústeres es superior al de 11 por su capacidad de rescatar temas específicos de administración, la redundancia es todavía demasiado alta para una arquitectura de información óptima. Esto justifica la necesidad de explorar $k=15$ para intentar aislar definitivamente los temas de seguridad (conducción y salud reproductiva) que aún no pertenecen a una sección independiente y limpia.